In [ ]:
#Necessary Dependencies 
import sys
sys.path.append("..")

import importlib
import tensorflow as tf
import DataModules.DataAugmentation
importlib.reload(DataModules.DataAugmentation)

from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Flatten, Dropout
from keras.applications.nasnet import NASNetLarge
from keras.applications.nasnet import preprocess_input
from DataModules.DataAugmentation import get_datasets

In [ ]:
train_dataset1, test_dataset1, train_dataset2, test_dataset2, train_dataset3, test_dataset3, train_dataset4, test_dataset4, data_augmentation = get_datasets()

In [ ]:
#The Model for RNFL_DM

nasnet_rnfl_dm = NASNetLarge(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in nasnet_rnfl_dm.layers:
    layer.trainable = False

x1 = nasnet_rnfl_dm.output
x1 = data_augmentation()(x1)
x1 = GlobalAveragePooling2D()(x1)
x1 = Dense(10, activation='relu')(x1)
x1 = Dropout(.5)(x1)
x1 = Flatten()(x1)
nasnet_prediction1 = Dense(1, activation='sigmoid')(x1)

# create a model object
nasnet_model1 = Model(inputs=nasnet_rnfl_dm.inputs, outputs=nasnet_prediction1)

In [ ]:
#The Model for RNFL_TM

nasnet_rnfl_tm = NASNetLarge(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in nasnet_rnfl_tm.layers:
    layer.trainable = False

x2 = nasnet_rnfl_tm.output
x2 = data_augmentation()(x2)
x2 = GlobalAveragePooling2D()(x2)
x2 = Dense(10, activation='relu')(x2)
x2 = Dropout(.5)(x2)
x2 = Flatten()(x2)
nasnet_prediction2 = Dense(1, activation='sigmoid')(x2)

# create a model object
nasnet_model2 = Model(inputs=nasnet_rnfl_tm.inputs, outputs=nasnet_prediction2)

In [ ]:
#The Model for GCIPL_DM

nasnet_gcipl_dm = NASNetLarge(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in nasnet_gcipl_dm.layers:
    layer.trainable = False

x3 = nasnet_gcipl_dm.output
x3 = data_augmentation()(x3)
x3 = GlobalAveragePooling2D()(x3)
x3 = Dense(10, activation='relu')(x3)
x3 = Dropout(.5)(x3)
x3 = Flatten()(x3)
nasnet_prediction3 = Dense(1, activation='sigmoid')(x3)

# create a model object
nasnet_model3 = Model(inputs=nasnet_gcipl_dm.inputs, outputs=nasnet_prediction3)

In [ ]:
#The Model for GCIPL_TM

nasnet_gcipl_tm = NASNetLarge(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in nasnet_gcipl_tm.layers:
    layer.trainable = False

x4 = nasnet_gcipl_tm.output
x4 = data_augmentation()(x4)
x4 = GlobalAveragePooling2D()(x4)
x4 = Dense(10, activation='relu')(x4)
x4 = Dropout(.5)(x4)
x4 = Flatten()(x4)
nasnet_prediction4 = Dense(1, activation='sigmoid')(x4)

# create a model object
nasnet_model4 = Model(inputs=nasnet_gcipl_tm.inputs, outputs=nasnet_prediction4)

In [ ]:
histories = {}

models = {
    "NASNet_RNFL(DM)": (nasnet_model1, train_dataset1, test_dataset1),
    "NASNet_RNFL(TM)": (nasnet_model2, train_dataset2, test_dataset2),
    "NASNet_GCIPL(DM)": (nasnet_model3, train_dataset3, test_dataset3),
    "NASNet_GCIPL(TM)": (nasnet_model4, train_dataset4, test_dataset4)
}

for name, (model, train_dataset, test_dataset) in models.items():
    tf.keras.backend.clear_session()
    print(f"Training {name}...")

    model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
    )
    
    histories[name] = model.fit(
        train_dataset,
        validation_data=test_dataset,
        epochs=150
    )